# Quality control notebook for Pasteur BioImage Analysis Course 2026

The notebook was adapted from Joanna Pylvänäinen.
---

In [ ]:
#@markdown ##Play the cell to connect your Google Drive to Colab

#@markdown * Click on the URL.

#@markdown * Sign in your Google Account.

#@markdown * Copy the authorization code.

#@markdown * Enter the authorization code.

#@markdown * Click on "Files" site on the right. Refresh the site. Your Google Drive folder should now be available here as "drive".

# mount user's Google Drive to Google Colab.
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
#@markdown ##Load dependencies

import os
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from tqdm import tqdm
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd
from tabulate import tabulate
import ipywidgets as widgets
from IPython.display import display, clear_output


In [ ]:
#@markdown ##Conduct QC on test data

# Define paths
gt_labels = '/content/gdrive/MyDrive/BIAC_Course_2026/2-test/2-label'  #@param {type:"string"}
predicted_labels = '/content/gdrive/MyDrive/BIAC_Course_2026/3-model/BIAC_100/results/BIAC_100_1/per_image_instances'  #@param {type:"string"}
results_folder = '/content/gdrive/MyDrive/BIAC_Course_2026/3-model/BIAC_100/results/BIAC_100_1' #@param {type:"string"}

# Create the QC folder inside the results folder
QC_folder = os.path.join(results_folder, "QC")
os.makedirs(QC_folder, exist_ok=True) # Create the folder if it doesn't exist

def load_label_images_with_names(folder_path):
    file_names = sorted(f for f in os.listdir(folder_path) if f.lower().endswith(('.tif', '.tiff', '.png', '.jpg', '.jpeg')))
    return [(fn, imread(os.path.join(folder_path, fn))) for fn in file_names]

# Load data
gt_data = load_label_images_with_names(gt_labels)
pred_data = load_label_images_with_names(predicted_labels)

assert len(gt_data) == len(pred_data), "Mismatch in number of GT and predicted images."

# ---- Initialize PDF for saving visualizations ---- #
pdf_path = os.path.join(QC_folder, "QC_visualizations.pdf")
pdf = PdfPages(pdf_path)

# ---- Evaluate and visualize ---- #
records = []
print("\nEvaluating and saving visualizations...\n")
df_res = pd.read_csv(os.path.join(results_folder, "test_results_metrics.csv")).set_index("file")
df_res = df_res[[col for col in df_res.columns if not any(th in col for th in ["0.3", "0.5", "0.75"]) or "0.5" in col]]

for (gt_name, gt_img), (pred_name, pred_img) in tqdm(zip(gt_data, pred_data), total=len(gt_data), desc="Processing"):
    assert gt_name == pred_name, f"Mismatched files: {gt_name} vs {pred_name}"

    # Calculate measured IoU for binary masks
    gt_mask = (gt_img > 0).astype(np.uint8) * 255
    pred_mask = (pred_img > 0).astype(np.uint8) * 255
    intersection = np.logical_and(gt_mask, pred_mask)
    union = np.logical_or(gt_mask, pred_mask)
    iou_score = np.sum(intersection) / np.sum(union) if np.sum(union) > 0 else 0

    df_res.loc[gt_name, "measured_IoU"] = iou_score  # Add measured IoU to the record

    # Create RGB images with black background
    gt_rgb = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    gt_rgb[gt_mask > 0] = [255, 0, 0]  # bright red

    pred_rgb = np.zeros((*pred_mask.shape, 3), dtype=np.uint8)
    pred_rgb[pred_mask > 0] = [0, 255, 0]  # bright green

    # Overlay: green + red, overlap becomes yellow
    overlay_rgb = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    overlay_rgb[gt_mask > 0, 0] = 255   # red channel
    overlay_rgb[pred_mask > 0, 1] = 255  # green channel

    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    fig.suptitle(f'{gt_name} — IoU: {iou_score:.3f}', fontsize=14)

    axes[0].imshow(gt_rgb)
    axes[0].set_title('Ground Truth')
    axes[0].axis('off')

    axes[1].imshow(pred_rgb)
    axes[1].set_title('Prediction')
    axes[1].axis('off')

    axes[2].imshow(overlay_rgb)
    axes[2].set_title('Overlay')
    axes[2].axis('off')

    plt.tight_layout()

    # Save figure to PDF
    pdf.savefig()
    plt.close()

# Save PDF
pdf.close()

# ---- Save results as CSV ---- #
df = df_res.reset_index(drop=False)
cols = ['file', 'measured_IoU'] + [col for col in df.columns if col not in ['file', 'measured_IoU']]
df = df[cols]  # Reorder columns
csv_path = os.path.join(QC_folder, "QC_metrics.csv")
df.to_csv(csv_path, index=False)
df = df.drop(['iou (f channel)', 'iou (c channel)'], axis=1)

# ---- Print final message ---- #
print("\n✅ Visualizations saved to:", pdf_path)
print("✅ Metrics CSV saved to:", csv_path)

In [ ]:
#@markdown ##Show QC metrics

print("\nFinal quality control summary:\n")

# ---- ADD PULLDOWN MENU FOR IMAGE PREVIEW ---- #

def show_image_preview(idx, df):
    gt_name, gt_img = gt_data[idx]
    _, pred_img = pred_data[idx]
    gt_mask = (gt_img > 0).astype(np.uint8) * 255
    pred_mask = (pred_img > 0).astype(np.uint8) * 255

    gt_rgb = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    gt_rgb[gt_mask > 0] = [255, 0, 0]  # bright red

    pred_rgb = np.zeros((*pred_mask.shape, 3), dtype=np.uint8)
    pred_rgb[pred_mask > 0] = [0, 255, 0]  # bright green

    # Overlay: green + red, overlap becomes yellow
    overlay_rgb = np.zeros((*gt_mask.shape, 3), dtype=np.uint8)
    overlay_rgb[gt_mask > 0, 0] = 255   # red channel
    overlay_rgb[pred_mask > 0, 1] = 255  # green channel

    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    iou_score = df.query('file == @gt_name')['measured_IoU'].values[0]
    fig.suptitle(f'{gt_name} — IoU: {iou_score:.3f}', fontsize=14)

    axes[0].imshow(gt_rgb)
    axes[0].set_title('Ground Truth')
    axes[0].axis('off')

    axes[1].imshow(pred_rgb)
    axes[1].set_title('Prediction')
    axes[1].axis('off')

    axes[2].imshow(overlay_rgb)
    axes[2].set_title('Overlay')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

# Create dropdown widget
image_names = [name for name, _ in gt_data]
dropdown = widgets.Dropdown(
    options=[(name, idx) for idx, name in enumerate(image_names)],
    value=0,
    description='Select Image:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='50%')
)

output = widgets.Output()

def on_dropdown_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        with output:
            clear_output(wait=True)
            show_image_preview(change['new'])

dropdown.observe(on_dropdown_change)

# Display dropdown and initial image
display(dropdown)
display(output)
with output:
    show_image_preview(0, df)

di = {'file': 'Filename', 'measured_IoU': 'Intersection over union', '0.5 TH fp': 'False positives', '0.5 TH tp': 'True positives', '0.5 TH fn': 'False negatives', '0.5 TH precision': 'Precision', '0.5 TH recall': 'Recall', '0.5 TH accuracy': 'Accuracy', '0.5 TH f1': 'F1-score', '0.5 TH n_true': 'Nr of ground truth objects', '0.5 TH n_pred': 'Nr of predicted objects', '0.5 TH mean_true_score': 'Mean true score', '0.5 TH mean_matched_score': 'Mean matched score', '0.5 TH panoptic_quality': 'Panoptic quality'}
df.loc['mean'] = df.mean(numeric_only=True)
df.rename(columns=di).round(3)
